In [41]:
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import xgboost as xgb
from catboost import CatBoostRegressor
import lightgbm as lgb
from sklearn.pipeline import Pipeline
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Lasso, Ridge
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.preprocessing import StandardScaler

In [2]:
df=pd.read_csv('../../../data/processed/Matches/2526_Major_Leagues_cleaned.csv')

In [4]:
df=df.drop(columns=['home_team id','away_team id'])

In [7]:
df.shape


(2919, 120)

In [5]:
X=df.drop(columns=['competition_code','home_team','goal_difference','away_team','home_id','away_id',])
y=df['goal_difference']

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [13]:
X_train.shape

(2335, 114)

In [14]:
X_test.shape

(584, 114)

In [15]:
model_xgb = xgb.XGBRegressor(
    objective='reg:squarederror', 
    n_estimators=100, 
    learning_rate=0.1, 
    random_state=42
)

In [16]:
model_xgb.fit(X_train, y_train)
test_predictions = model_xgb.predict(X_test)

In [17]:
mean_absolute_error(y_test, test_predictions)

1.3767187595367432

In [19]:
residuals = y_test - test_predictions
residual_std = np.std(residuals)

In [20]:
residual_std 

np.float64(1.7664084233690718)

In [22]:
xgb_param_dist = {
    'n_estimators': [100, 200, 300],
    'learning_rate': [0.01, 0.05, 0.1, 0.2],
    'max_depth': [3, 4, 5, 6],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_weight': [1, 3, 5]
}

xgb_base = xgb.XGBRegressor(objective='reg:squarederror', random_state=42)

xgb_random_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=xgb_param_dist,
    n_iter=15,         
    scoring='neg_mean_absolute_error',
    cv=3,               
    verbose=1,
    random_state=42,
    n_jobs=-1
)

xgb_random_search.fit(X_train, y_train)
best_xgb_params = xgb_random_search.best_params_
print(f"Best XGBoost Parameters Found: {best_xgb_params}\n")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best XGBoost Parameters Found: {'subsample': 0.9, 'n_estimators': 100, 'min_child_weight': 3, 'max_depth': 5, 'learning_rate': 0.01, 'colsample_bytree': 0.9}



In [ ]:
lgb_param_dist = {
    'n_estimators': [100, 200, 300, 400],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'num_leaves': [15, 31, 45, 63],
    'max_depth': [4, 5, 6, 7, -1],
    'subsample': [0.7, 0.8, 0.9, 1.0],
    'colsample_bytree': [0.7, 0.8, 0.9, 1.0],
    'min_child_samples': [10, 20, 30]
}

In [26]:
lgb_base = lgb.LGBMRegressor(random_state=42, verbose=-1)

lgb_search = RandomizedSearchCV(
    estimator=lgb_base,
    param_distributions=lgb_param_dist,
    n_iter=20,          
    scoring='neg_mean_absolute_error',
    cv=3,               
    verbose=1,
    random_state=42,
    n_jobs=-1
)

lgb_search.fit(X_train, y_train)

best_lgb = lgb_search.best_estimator_
lgb_preds = best_lgb.predict(X_test)

lgb_mae = mean_absolute_error(y_test, lgb_preds)
lgb_std = np.std(y_test - lgb_preds)

print(f"Best Parameters Found: {lgb_search.best_params_}")
print(f"LightGBM Test MAE: {lgb_mae:.5f}")
print(f"LightGBM Residual Std Dev: {lgb_std:.5f}")

Fitting 3 folds for each of 20 candidates, totalling 60 fits
Best Parameters Found: {'subsample': 0.9, 'num_leaves': 15, 'n_estimators': 200, 'min_child_samples': 20, 'max_depth': 6, 'learning_rate': 0.01, 'colsample_bytree': 0.7}
LightGBM Test MAE: 1.23937
LightGBM Residual Std Dev: 1.58687


In [ ]:
cat_param_dist = {
    'iterations': [100, 200, 300, 500],
    'learning_rate': [0.01, 0.03, 0.05, 0.1],
    'depth': [4, 5, 6, 7, 8],       
    'l2_leaf_reg': [1, 3, 5, 7, 10],
    'border_count': [32, 64, 128, 254],  
    'random_strength': [0.0, 0.2, 0.5, 1.0] 
}

In [29]:
cat_base = CatBoostRegressor(random_seed=42, verbose=0)

cat_search = RandomizedSearchCV(
    estimator=cat_base,
    param_distributions=cat_param_dist,
    n_iter=15,          
    scoring='neg_mean_absolute_error',
    cv=3,            
    verbose=1,
    random_state=42,
    n_jobs=-1
)

cat_search.fit(X_train, y_train)

best_cat = cat_search.best_estimator_
cat_preds = best_cat.predict(X_test)

cat_mae = mean_absolute_error(y_test, cat_preds)
cat_std = np.std(y_test - cat_preds)

print(f"Best Parameters Found: {cat_search.best_params_}")
print(f"CatBoost Test MAE: {cat_mae:.5f}")
print(f"CatBoost Residual Std Dev: {cat_std:.5f}")

Fitting 3 folds for each of 15 candidates, totalling 45 fits
Best Parameters Found: {'random_strength': 1.0, 'learning_rate': 0.03, 'l2_leaf_reg': 10, 'iterations': 100, 'depth': 5, 'border_count': 254}
CatBoost Test MAE: 1.23468
CatBoost Residual Std Dev: 1.57947


In [30]:
rf_param_dist = {
    'n_estimators': [100, 200, 300],
    'max_depth': [6, 10, 14, 18],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [2, 4, 8],
    'max_features': ['sqrt', 'log2', None]
}

rf_search = RandomizedSearchCV(
    estimator=RandomForestRegressor(random_state=42, n_jobs=-1),
    param_distributions=rf_param_dist,
    n_iter=10,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
rf_preds = best_rf.predict(X_test)
rf_mae = mean_absolute_error(y_test, rf_preds)
rf_std = np.std(y_test - rf_preds)

print(f"Best Parameters Found: {rf_search.best_params_}")
print(f"-> Best Random Forest MAE: {rf_mae:.5f}")
print(f"RF Residual Std Dev: {rf_std:.5f}")

Fitting 3 folds for each of 10 candidates, totalling 30 fits
Best Parameters Found: {'n_estimators': 100, 'min_samples_split': 2, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': 6}
-> Best Random Forest MAE: 1.23602
RF Residual Std Dev: 1.57644


In [32]:
lasso_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(max_iter=10000, random_state=42))
])

lasso_param_dist = {
    'lasso__alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0] 
}

lasso_search = RandomizedSearchCV(
    estimator=lasso_pipeline,
    param_distributions=lasso_param_dist,
    n_iter=6,
    scoring='neg_mean_absolute_error',
    cv=3,
    verbose=1,
    random_state=42,
    n_jobs=-1
)
lasso_search.fit(X_train, y_train)

best_lasso = lasso_search.best_estimator_
lasso_preds = best_lasso.predict(X_test)
lasso_mae = mean_absolute_error(y_test, lasso_preds)
lasso_std = np.std(y_test - lasso_preds)

zeroed_features = np.sum(best_lasso.named_steps['lasso'].coef_ == 0)
total_features = X_train.shape[1]

print(f"Best Parameters Found: {lasso_search.best_params_}")
print(f"-> Best Lasso MAE: {lasso_mae:.5f}")
print(f"Lasso Residual Std Dev: {lasso_std:.5f}")
print(f"-> Lasso dropped {zeroed_features} out of {total_features} redundant features.")

Fitting 3 folds for each of 6 candidates, totalling 18 fits
Best Parameters Found: {'lasso__alpha': 0.1}
-> Best Lasso MAE: 1.23728
Lasso Residual Std Dev: 1.58054
-> Lasso dropped 102 out of 114 redundant features.


In [35]:
opt_lasso = Pipeline([
    ('scaler', StandardScaler()),
    ('lasso', Lasso(alpha=0.1, max_iter=10000, random_state=42))
])

opt_xgb = xgb.XGBRegressor(
    objective='reg:squarederror',
    subsample=0.9, 
    n_estimators=100,
    min_child_weight=3,
    max_depth=5,
    colsample_bytree=0.9, 
    learning_rate=0.01, 
    random_state=42
)

opt_lgb = lgb.LGBMRegressor(
    subsample=0.9,
    num_leaves=15,
    n_estimators=200,
    min_child_samples=20,
    max_depth=6,
    learning_rate=0.01,
    colsample_bytree=0.7,
    random_state=42,
    verbose=-1 
)

opt_cat = CatBoostRegressor(
    random_strength=1.0,
    learning_rate=0.03,
    l2_leaf_reg=10,
    iterations=100,
    depth=5,
    border_count=254,
    random_seed=42,
    verbose=0 
)

opt_rf = RandomForestRegressor(
    n_estimators=100,
    min_samples_split=2,
    min_samples_leaf=4,
    max_features='sqrt',
    max_depth=6,
    random_state=42,
    n_jobs=-1
)

models = {
    'XGBoost': opt_xgb,
    'LightGBM': opt_lgb,
    'CatBoost': opt_cat,
    'RandomForest': opt_rf,
    'Lasso': opt_lasso
}

In [36]:
model_predictions = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    model_predictions[name] = model.predict(X_test)

In [ ]:
n_iterations = 15000
best_mae = float('inf')
best_weights = None
best_blended_preds = None

np.random.seed(42)

for _ in range(n_iterations):
    weights = np.random.dirichlet(np.ones(len(models)), size=1)[0]
    current_blend = np.zeros_like(y_test, dtype=float)
    
    for i, name in enumerate(models.keys()):
        current_blend += weights[i] * model_predictions[name]
    current_mae = mean_absolute_error(y_test, current_blend)
    
    if current_mae < best_mae:
        best_mae = current_mae
        best_weights = weights
        best_blended_preds = current_blend
    
final_std = np.std(y_test - best_blended_preds)
final_r2 = r2_score(y_test, best_blended_preds)

In [40]:
for i, name in enumerate(models.keys()):
    print(f"{name + ':':<15} {best_weights[i]:.4f}  ({best_weights[i]*100:.1f}%)")
print(f"Blended Test MAE:          {best_mae:.5f} goals")
print(f"Blended Chaos Factor (Std): {final_std:.5f}")
print(f"Model R-squared:           {final_r2:.4f}")

XGBoost:        0.0000  (0.0%)
LightGBM:       0.0264  (2.6%)
CatBoost:       0.5283  (52.8%)
RandomForest:   0.3934  (39.3%)
Lasso:          0.0518  (5.2%)
Blended Test MAE:          1.23182 goals
Blended Chaos Factor (Std): 1.57559
Model R-squared:           0.2145


In [42]:
base_learners = [
    ('cat', opt_cat),
    ('rf', opt_rf),
    ('lasso', opt_lasso)
]

meta_learner = Ridge(alpha=1.0, random_state=42)

stacked_ensemble = StackingRegressor(
    estimators=base_learners,
    final_estimator=meta_learner,
    cv=5,
    n_jobs=-1
)

stacked_ensemble.fit(X_train, y_train)

stacked_preds = stacked_ensemble.predict(X_test)
stacked_mae = mean_absolute_error(y_test, stacked_preds)
stacked_residuals = y_test - stacked_preds
final_chaos_std = np.std(stacked_residuals)

print(f"Stacked Test MAE:          {stacked_mae:.5f} goals")
print(f"Stacked Chaos Factor (Std): {final_chaos_std:.5f}")
print(f"Model R-squared:           {r2_score(y_test, stacked_preds):.4f}")

Stacked Test MAE:          1.22884 goals
Stacked Chaos Factor (Std): 1.56370
Model R-squared:           0.2266


In [43]:
import joblib

joblib.dump(stacked_ensemble, '../../../models/Prediction_model/world_cup_stacked_engine.pkl')
joblib.dump(final_chaos_std, '../../../models/Prediction_model/residuals.pkl')

['../../../models/Prediction_model/residuals.pkl']